In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, col, rand, when
from pyspark import StorageLevel
import time

#1. Otimize o Shuffle Partitions
#Regra prática: 2-3x o número de cores. Se tem 8 cores, 24-32 partitions
#Ajuste conforme tamanho dos dados: ~128MB por partition no shuffle
spark = SparkSession.builder \
    .appName("OtimizacaoCompleta") \
    .config("spark.sql.shuffle.partitions", "32") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .getOrCreate()

#Primeiro: gerar o dataset de teste
print("Gerando datasets de teste...")

#2. Ajuste o tamanho dos blocos (block size)
#Para arquivos Parquet: controla com maxPartitionBytes. Default 128MB
spark.conf.set("spark.sql.files.maxPartitionBytes", "134217728") # 128MB
spark.conf.set("spark.sql.files.openCostInBytes", "4194304") # 4MB custo de abrir arquivo

#Dataset grande - tabela fato
df_vendas = spark.range(0, 10000000) \
    .withColumn("id_cliente", (rand() * 100000).cast("int")) \
    .withColumn("id_produto", (rand() * 50000).cast("int")) \
    .withColumn("valor", rand() * 1000) \
    .withColumn("data", when(rand() < 0.3, "2025-01-01").otherwise("2025-01-02")) \
    .repartition(32, "id_cliente") # 2. Reparticiona na escrita pra evitar arquivos pequenos

#Dataset pequeno - tabela dimensão
df_clientes = spark.range(0, 100000) \
    .withColumn("nome", col("id").cast("string")) \
    .withColumn("regiao", when(rand() < 0.2, "Sul")
                           .when(rand() < 0.4, "Sudeste")
                           .when(rand() < 0.6, "Norte")
                           .otherwise("Nordeste")) \
    .withColumnRenamed("id", "id_cliente")

#3. Use o cache de dados
#Só cacheia o que vai reusar. MEMORY_AND_DISK é mais seguro que MEMORY_ONLY
df_clientes.cache() # ou .persist(StorageLevel.MEMORY_AND_DISK)
df_clientes.count() # Ação pra materializar o cache

#4. Otimize as junções (joins)
#Broadcast para tabela pequena < 10MB ou spark.sql.autoBroadcastJoinThreshold
#Salting se tiver skew, mas aqui usamos broadcast que é melhor pra esse caso
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760") # 10MB

inicio = time.time()

df_resultado = df_vendas.join(
    broadcast(df_clientes), # 4. Força broadcast join
    "id_cliente",
    "inner"
).filter(col("regiao") == "Sudeste") \
 .groupBy("id_produto") \
 .agg({"valor": "sum", "id_cliente": "count"}) \
 .withColumnRenamed("sum(valor)", "total_vendas") \
 .withColumnRenamed("count(id_cliente)", "qtd_transacoes")

#3. Cache do resultado se for usar múltiplas vezes
df_resultado.persist(StorageLevel.MEMORY_AND_DISK)

#Ação final
df_resultado.show(20)
print(f"Tempo de execução: {time.time() - inicio:.2f}s")

#5. Monitore e ajuste do paralelismo e garbage collection (GC)
#Configs importantes pra colocar no spark-submit ou SparkSession:
"""
--conf spark.executor.memory=4g
--conf spark.executor.cores=4
--conf spark.default.parallelism=32
--conf spark.memory.fraction=0.8
--conf spark.memory.storageFraction=0.3
--conf spark.executor.extraJavaOptions="-XX:+UseG1GC -XX:InitiatorHeapOccupancyPercent=35 -verbose:gc -XX:+PrintGCDetails"
"""

#Verificar plano físico e se o broadcast funcionou
df_resultado.explain("formatted")

#6. Use o Spark SQL
#Mesmo resultado usando SQL - Catalyst otimiza igual, mas às vezes é mais legível
df_vendas.createOrReplaceTempView("vendas")
df_clientes.createOrReplaceTempView("clientes")

df_sql = spark.sql("""
    SELECT /*+ BROADCAST(c) */
        v.id_produto,
        SUM(v.valor) as total_vendas,
        COUNT(v.id_cliente) as qtd_transacoes
    FROM vendas v
    INNER JOIN clientes c ON v.id_cliente = c.id_cliente
    WHERE c.regiao = 'Sudeste'
    GROUP BY v.id_produto
""")

df_sql.show(20)

#Limpar cache quando terminar
spark.catalog.clearCache()
spark.stop()
# Onde cada otimização entra
'''
Otimização Onde está no código
1 Shuffle Partitions `spark.sql.shuffle.partitions=32` + AQE habilitado
2 Tamanho dos blocos `maxPartitionBytes=128MB` + `repartition(32)` na geração
3 Cache de dados `df_clientes.cache()` e `df_resultado.persist()`
4 Otimize joins `broadcast(df_clientes)` + hint `/*+ BROADCAST(c) */` no SQL
5 Paralelismo e GC Comentário com configs de `--conf` pra usar no submit
6 Spark SQL Último bloco com `spark.sql()` fazendo a mesma lógica
*Dicas rápidas pra produção:*

1. *Shuffle*: Deixe AQE ligado. Ele ajusta o nº de partitions sozinho após o Spark 3.0
2. *Block size*: Se ler de S3, usa 128MB. HDFS aguenta 256MB tranquilo
3. *Cache*: Só use se o DF for lido 2+ vezes. Senão só ocupa memória
4. *Joins*: Sempre filtra antes do join. Broadcast só pra tabelas <200MB
5. *GC*: G1GC é padrão no Spark 3+. Monitore com Spark UI na aba Executors > GC Time
'''

Gerando datasets de teste...
+----------+--------------+------------------+
|id_produto|qtd_transacoes|      total_vendas|
+----------+--------------+------------------+
|     21653|            55| 21583.39367448556|
|     32878|            68| 30550.09666110041|
|     42124|            52|24870.592138472828|
|     28571|            60|25947.242575674336|
|       273|            52|23209.418764338032|
|      6778|            74|35772.360361973275|
|     23991|            57|27704.260882417766|
|     45134|            70| 32381.84733704237|
|     43053|            63|31131.866868408797|
|     36525|            67|32511.077221420266|
|     22026|            62| 29811.89322701575|
|      8493|            66| 32386.10683636835|
|      4512|            54| 24633.37658751843|
|      2383|            73|39029.703868612785|
|     32314|            60| 32922.88477754201|
|     32320|            76| 40235.46214541708|
|     27955|            73| 37983.31970514436|
|     26398|            64|2999

'\nOtimização Onde está no código\n1 Shuffle Partitions `spark.sql.shuffle.partitions=32` + AQE habilitado\n2 Tamanho dos blocos `maxPartitionBytes=128MB` + `repartition(32)` na geração\n3 Cache de dados `df_clientes.cache()` e `df_resultado.persist()`\n4 Otimize joins `broadcast(df_clientes)` + hint `/*+ BROADCAST(c) */` no SQL\n5 Paralelismo e GC Comentário com configs de `--conf` pra usar no submit\n6 Spark SQL Último bloco com `spark.sql()` fazendo a mesma lógica\n*Dicas rápidas pra produção:*\n\n1. *Shuffle*: Deixe AQE ligado. Ele ajusta o nº de partitions sozinho após o Spark 3.0\n2. *Block size*: Se ler de S3, usa 128MB. HDFS aguenta 256MB tranquilo\n3. *Cache*: Só use se o DF for lido 2+ vezes. Senão só ocupa memória\n4. *Joins*: Sempre filtra antes do join. Broadcast só pra tabelas <200MB\n5. *GC*: G1GC é padrão no Spark 3+. Monitore com Spark UI na aba Executors > GC Time\n'